# Prompt 1A — Clean Raw HMDA Data

## 1. Objective

This notebook reports the completed Prompt 1A population build. It uses saved reports only. It does not rebuild data, select a sample, engineer features, or fit a model.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
REPORTS = ROOT / 'outputs' / 'reports'

def load_json(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))

source_report = load_json('prompt1a_source_report.json')
legacy_rules = load_json('legacy_cleaning_rules.json')
recovery = load_json('prompt1a_count_recovery.json')
schema_report = load_json('prompt1a_schema.json')
run_summary = load_json('prompt1a_run_summary.json')
manifest = load_json('prompt1a_partition_manifest.json')
cleaning = pd.read_csv(REPORTS / 'prompt1a_cleaning_report.csv')
outliers = pd.read_csv(REPORTS / 'prompt1a_outlier_review.csv')
print('Saved Prompt 1A reports loaded. No source-data rebuild was run.')

Saved Prompt 1A reports loaded. No source-data rebuild was run.


## 2. Source data

The extracted 2017 nationwide HMDA CSV was the only processing source. The ZIP stayed preserved and was not extracted again. The recovery pass checked the source identity in the same stream used for counting.

In [2]:
source_table = pd.DataFrame([{
    'raw_rows': recovery['reconciliation']['raw_rows'],
    'size_bytes': source_report['size_bytes'],
    'encoding': source_report['encoding'],
    'delimiter': source_report['delimiter'],
    'sha256': recovery['source_identity']['observed_sha256'],
    'hash_match': recovery['source_identity']['sha256_match'],
    'path_size_mtime_unchanged': all(recovery['source_identity'][key] for key in ['path_unchanged', 'size_unchanged', 'modification_time_unchanged'])
}])
display(source_table)

,raw_rows,size_bytes,encoding,delimiter,sha256,hash_match,path_size_mtime_unchanged
0,14285496,11237068086,UTF-8 (no BOM detected; strict sampled decodin...,",",dd35f6a877c5882bbe7260ce65ca842b18ca6aa16514cb...,True,True


## 3. Legacy cleaning rules

Two read-only legacy notebooks defined the original scientific intent. Prompt 1A kept their required fields, numeric rules, categorical codes, target units, and outlier policy. The newer specification clearly limited missing-value removal to 16 required fields.

In [3]:
legacy_summary = pd.DataFrame([
    {'item': 'Required fields', 'value': len(legacy_rules['required_fields'])},
    {'item': 'Retained base fields', 'value': len(legacy_rules['final_retained_base_fields'])},
    {'item': 'Validated categorical code fields', 'value': len(legacy_rules['valid_categorical_codes'])},
    {'item': 'Target', 'value': legacy_rules['target_definition']['field']},
    {'item': 'Target unit', 'value': legacy_rules['target_definition']['unit']},
    {'item': 'Duplicate handling', 'value': legacy_rules['duplicate_handling']}
])
display(legacy_summary)

,item,value
0,Required fields,16
1,Retained base fields,30
2,Validated categorical code fields,15
3,Target,loan_amount_000s
4,Target unit,thousands of U.S. dollars
5,Duplicate handling,Deferred to Prompt 1B. The legacy notebooks do...


## 4. Regression population

Eligible rows have action codes 1, 2, or 8. The target is `loan_amount_000s`, measured in thousands of U.S. dollars. `action_taken_code` is retained for audit only, and the constant legacy approval feature is not present.

In [4]:
population_definition = pd.DataFrame([{
    'eligible_action_codes': '1, 2, 8',
    'target': 'loan_amount_000s',
    'target_unit': 'thousands of U.S. dollars',
    'audit_field': 'action_taken_code',
    'loan_approved_retained': False
}])
display(population_definition)

,eligible_action_codes,target,target_unit,audit_field,loan_approved_retained
0,"1, 2, 8",loan_amount_000s,thousands of U.S. dollars,action_taken_code,False


## 5. Exclusive population-flow counters

Each row appears in only its first applicable removal category. The final retained row is not a removal. The counts reconcile exactly to the complete source.

In [5]:
flow_table = cleaning.copy()
flow_table['percentage_of_raw_rows'] = flow_table['percentage_of_raw_rows'].round(4)
display(flow_table)
removed_total = int(flow_table['exclusive_removed_rows'].sum())
retained_total = int(flow_table.loc[flow_table['rule_name'] == 'retained', 'retained_rows'].iloc[0])
reconciliation_table = pd.DataFrame([{
    'raw_rows': recovery['reconciliation']['raw_rows'],
    'exclusive_removals': removed_total,
    'retained_rows': retained_total,
    'removals_plus_retained': removed_total + retained_total,
    'exact_match': recovery['reconciliation']['equation_matches_raw']
}])
display(reconciliation_table)

,rule_order,rule_name,exclusive_removed_rows,retained_rows,percentage_of_raw_rows,cumulative_rows_remaining,notes
0,1,malformed_csv_structure,0,0,0.0000,14285496,Structurally malformed CSV rows; first exclusi...
1,2,ineligible_action,6500536,0,45.5044,7784960,"Action code is not 1, 2, or 8."
2,3,required_missing_like,1305782,0,9.1406,6479178,At least one of the 16 required fields is miss...
3,4,invalid_numeric_parse_or_nonfinite,0,0,0.0000,6479178,Required numeric parsing failed or produced in...
4,5,invalid_numeric_range,7548,0,0.0528,6471630,A numeric value violates the fixed positive or...
5,6,invalid_categorical_code,0,0,0.0000,6471630,A present categorical code is outside its fixe...
6,7,retained,0,6471630,45.3021,6471630,Complete cleaned eligible population; no dedup...


,raw_rows,exclusive_removals,retained_rows,removals_plus_retained,exact_match
0,14285496,7813866,6471630,14285496,True


## 6. Required missing-value removal

A row was removed at this stage when at least one required field contained a fixed missing-like token. The exclusive total is smaller than the sum of column detections because one row may fail several required fields.

In [6]:
missing_diagnostics = pd.DataFrame(
    recovery['non_exclusive_diagnostic_counts']['required_missing_like'].items(),
    columns=['required_field', 'non_exclusive_detection_count']
).sort_values('non_exclusive_detection_count', ascending=False)
display(pd.DataFrame([{'exclusive_rows_removed': recovery['exact_exclusive_counters']['required_missing_like']}]))
display(missing_diagnostics)

,exclusive_rows_removed
0,1305782


,required_field,non_exclusive_detection_count
1,msamd_name,899125
2,msamd,897615
0,applicant_income_000s,454819
4,number_of_owner_occupied_units,63226
5,number_of_1_to_4_family_units,63226
8,minority_population,63226
7,population,63205
3,tract_to_msamd_income,63205
9,hud_median_family_income,63205
6,census_tract_number,63204


## 7. Numeric invalid-value removal

Numeric parsing rejected malformed and non-finite values. The range stage then required positive values, except that minority population could range from 0 to 100. Positive statistical extremes were kept.

In [7]:
numeric_exclusive = pd.DataFrame([
    {'rule': 'invalid_numeric_parse_or_nonfinite', 'exclusive_rows_removed': recovery['exact_exclusive_counters']['invalid_numeric_parse_or_nonfinite']},
    {'rule': 'invalid_numeric_range', 'exclusive_rows_removed': recovery['exact_exclusive_counters']['invalid_numeric_range']}
])
range_diagnostics = pd.DataFrame(
    recovery['non_exclusive_diagnostic_counts']['invalid_numeric_range'].items(),
    columns=['numeric_field', 'non_exclusive_range_detection_count']
).sort_values('non_exclusive_range_detection_count', ascending=False)
display(numeric_exclusive)
display(range_diagnostics)

,rule,exclusive_rows_removed
0,invalid_numeric_parse_or_nonfinite,0
1,invalid_numeric_range,7548


,numeric_field,non_exclusive_range_detection_count
5,tract_to_msamd_income,6763
6,number_of_owner_occupied_units,641
7,number_of_1_to_4_family_units,487
2,population,112
3,minority_population,0
1,applicant_income_000s,0
0,loan_amount_000s,0
4,hud_median_family_income,0


## 8. Invalid category-code removal

Present categorical codes were checked against the fixed allowed sets before redundant coded fields were dropped. Missing optional codes were not treated as invalid.

In [8]:
category_diagnostics = pd.DataFrame(
    recovery['non_exclusive_diagnostic_counts']['invalid_categorical_code'].items(),
    columns=['categorical_field', 'non_exclusive_invalid_code_count']
)
display(pd.DataFrame([{'exclusive_rows_removed': recovery['exact_exclusive_counters']['invalid_categorical_code']}]))
display(category_diagnostics)

,exclusive_rows_removed
0,0


,categorical_field,non_exclusive_invalid_code_count
0,agency_code,0
1,loan_type,0
2,property_type,0
3,loan_purpose,0
4,owner_occupancy,0
5,preapproval,0
6,action_taken,0
7,applicant_ethnicity,0
8,co_applicant_ethnicity,0
9,applicant_race_1,0


## 9. Outlier-retention policy

No IQR deletion, Z-score deletion, target cap, or winsorization was used. The table reports the complete retained population. Values equal to 1 and the largest 100 values were reviewed but kept when otherwise valid.

In [9]:
metric_order = ['minimum', 'p0_1', 'p0_5', 'p1', 'p5', 'p25', 'median', 'p75', 'p95', 'p99', 'p99_5', 'p99_9', 'maximum', 'mean', 'std']
complete_outliers = outliers[outliers['scope'] == 'complete_eligible_population']
outlier_summary = complete_outliers[complete_outliers['metric'].isin(metric_order)].pivot(index='column', columns='metric', values='value').reindex(columns=metric_order)
display(outlier_summary.round(3))
extreme_review = outliers[(outliers['metric'] == 'count_equal_1') | (outliers['scope'] == 'largest_100_values')]
display(extreme_review.reset_index(drop=True))

metric,minimum,p0_1,p0_5,p1,p5,p25,median,p75,p95,p99,p99_5,p99_9,maximum,mean,std
column,,,,,,,,,,,,,,,
applicant_income_000s,1.00,9.00,17.00,20.00,32.00,56.00,85.00,131.00,276.00,581.00,822.00,1909.371,268000.00,116.827,322.527
hud_median_family_income,18000.00,25900.00,42200.00,46000.00,53500.00,63600.00,70900.00,79300.00,104800.00,111200.00,114200.00,131500.000,131500.00,72920.689,14555.280
loan_amount_000s,1.00,2.00,5.00,6.00,33.00,126.00,201.00,310.00,590.00,1073.00,1400.00,2430.000,320000.00,249.030,452.920
minority_population,0.00,0.46,1.29,1.85,4.19,12.46,24.90,46.49,86.57,98.03,99.22,100.000,100.00,32.400,25.271
number_of_1_to_4_family_units,5.00,52.00,184.00,337.00,762.00,1332.00,1805.00,2409.00,3861.00,5924.00,7256.00,11372.000,25391.00,2002.170,1135.407
number_of_owner_occupied_units,2.00,82.00,183.00,247.00,481.00,950.00,1357.00,1859.00,2953.00,4659.00,5864.00,9575.000,19529.00,1508.724,925.686
population,30.00,919.00,1392.00,1625.00,2436.00,3951.00,5269.00,6903.00,11064.00,18303.00,22552.00,35527.000,53812.00,5860.649,3274.649
tract_to_msamd_income,3.71,28.12,37.39,42.14,58.63,88.29,110.66,137.46,192.97,249.29,283.67,346.990,507.47,116.414,42.377


,column,scope,metric,value
0,loan_amount_000s,complete_eligible_population,count_equal_1,3231.000000
1,loan_amount_000s,largest_100_values,count,100.000000
2,loan_amount_000s,largest_100_values,minimum,10000.000000
3,loan_amount_000s,largest_100_values,median,15000.000000
4,loan_amount_000s,largest_100_values,maximum,320000.000000
5,loan_amount_000s,largest_100_values,mean,57442.100000
6,loan_amount_000s,largest_100_values,std,82003.716378
7,applicant_income_000s,complete_eligible_population,count_equal_1,1064.000000
8,applicant_income_000s,largest_100_values,count,100.000000
9,applicant_income_000s,largest_100_values,minimum,12600.000000


## 10. Cleaned staging population

The complete eligible population is stored in existing compressed Parquet partitions. Prompt 1A did not globally deduplicate rows and did not create Development or Holdout data.

In [10]:
staging_population = pd.DataFrame([{
    'retained_count_from_recovery': recovery['retained_count_comparison']['count_pass'],
    'retained_count_from_partitions': recovery['retained_count_comparison']['partition_total'],
    'counts_equal': recovery['retained_count_comparison']['all_equal'],
    'global_deduplication_performed': False,
    'development_or_holdout_created': False
}])
display(staging_population)

,retained_count_from_recovery,retained_count_from_partitions,counts_equal,global_deduplication_performed,development_or_holdout_created
0,6471630,6471630,True,False,False


## 11. Staging schema and partitions

All partitions use one stable 30-field schema. Identifiers and labels remain strings, measured numeric fields are float64, and the audit action code is int8.

In [11]:
schema_table = pd.DataFrame(schema_report['fields'])
partition_table = pd.DataFrame(manifest)
partition_summary = pd.DataFrame([{
    'partitions': len(partition_table),
    'rows': int(partition_table['retained_row_count'].sum()),
    'storage_bytes': int(partition_table['output_size'].sum()),
    'unique_schema_digests': partition_table['schema_digest'].nunique(),
    'all_complete': bool(partition_table['completion_status'].eq('COMPLETE').all())
}])
display(schema_table)
display(partition_summary)

,name,type,nullable,reason
0,respondent_id,string,False,Lender identity and later lender ablation
1,agency_name,string,False,Legacy exact comparison and validated feature ...
2,loan_type_name,string,False,Legacy exact comparison and validated feature ...
3,property_type_name,string,False,Legacy exact comparison and validated feature ...
4,loan_purpose_name,string,False,Legacy exact comparison and validated feature ...
5,owner_occupancy_name,string,False,Legacy exact comparison and validated feature ...
6,loan_amount_000s,double,False,Regression target in thousands of U.S. dollars
7,preapproval_name,string,False,Legacy exact comparison and validated feature ...
8,msamd_name,string,False,"Legacy matching, geography, and validated feat..."
9,state_name,string,False,"Legacy matching, geography, and validated feat..."


,partitions,rows,storage_bytes,unique_schema_digests,all_complete
0,112,6471630,142659308,1,True


## 12. Runtime and storage

The recovery used one authorized complete count-only pass. It wrote no staging data. Peak memory is the highest observed process resident memory during the pass.

In [12]:
runtime_table = pd.DataFrame([{
    'initial_staging_build_seconds': run_summary['initial_staging_build_runtime_seconds'],
    'count_recovery_seconds': round(run_summary['count_recovery_runtime_seconds'], 3),
    'count_recovery_peak_memory_gib': round(run_summary['count_recovery_peak_memory_bytes'] / 1024**3, 3),
    'staging_storage_mib': round(run_summary['staging_storage_bytes'] / 1024**2, 3),
    'additional_raw_passes_used': run_summary['additional_complete_raw_passes_used'],
    'staging_rebuilt_in_recovery': run_summary['staging_rebuilt_during_recovery']
}])
display(runtime_table)

,initial_staging_build_seconds,count_recovery_seconds,count_recovery_peak_memory_gib,staging_storage_mib,additional_raw_passes_used,staging_rebuilt_in_recovery
0,249.8,229.868,5.102,136.051,1,False


## 13. Verification

The count equation, source identity, retained count, partition total, and staging immutability all passed. The manifest and staging snapshot digests were identical before and after recovery.

In [13]:
verification_table = pd.DataFrame([
    {'check': 'Source SHA-256 matches', 'passed': recovery['source_identity']['sha256_match']},
    {'check': 'Population equation reconciles', 'passed': recovery['reconciliation']['equation_matches_raw']},
    {'check': 'Retained count matches expected', 'passed': recovery['reconciliation']['retained_matches_expected']},
    {'check': 'Retained count matches partitions', 'passed': recovery['retained_count_comparison']['all_equal']},
    {'check': 'Manifest and staging unchanged', 'passed': recovery['staging_immutability']['unchanged']},
    {'check': 'Recovery status', 'passed': recovery['status'] == 'PASS'}
])
display(verification_table)

,check,passed
0,Source SHA-256 matches,True
1,Population equation reconciles,True
2,Retained count matches expected,True
3,Retained count matches partitions,True
4,Manifest and staging unchanged,True
5,Recovery status,True


## 14. Limitations

The original legacy workflow did not define one consistent global deduplication key. Prompt 1A therefore preserves all eligible cleaned rows. It also makes no fairness, causal, or model-performance claim. Statistical extremes are reported, not removed.

## 15. Prompt 1B handoff

Prompt 1A ends with the validated eligible population and exact cleaning counters. Prompt 1B may begin later from these frozen artifacts to define global deduplication, exclude Legacy rows, select final datasets, engineer features, and freeze data. None of those operations is performed here.